In [2]:
import json
import sys
import pathlib
source_dir = pathlib.Path.cwd().parent / 'software' / 'source'
if str(source_dir) not in sys.path:
    sys.path.append(str(source_dir))
import pathlib
import matplotlib.pyplot as plt
import numpy as np
from qick.asm_v2 import AveragerProgramV2
from qick_emu import QickEmu

In [3]:
CONFIG_FILE = "qick_config_216.json"
EMU_DIR = "emulator_output"

# Just pass the config file. QickEmu automatically creates the address map.
emu = QickEmu(qick_config_json=CONFIG_FILE)
soc = emu.make_soc(memdir=EMU_DIR)

print(f"Virtual SOC Ready: {len(soc.gens)} Gens, {len(soc.readouts)} Readouts.")

QICK library version mismatch: 0.2.366 remote (the board), 0.2.370 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


Virtual SOC Ready: 16 Gens, 10 Readouts.


In [4]:
class MultiPulseProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        ramp_len = self.cycles2us(20, gen_ch=gen_ch)
        self.add_gauss(ch=gen_ch, name="ramp", sigma=ramp_len/10, length=ramp_len, even_length=True)
        
        self.add_pulse(ch=gen_ch, name="flat1", ro_ch=ro_ch, 
                       style="flat_top", envelope="ramp", 
                       freq=cfg['freq'], length=0.1, phase=0, gain=1.0)

        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['freq'], gen_ch=gen_ch)
        self.send_readoutconfig(ch=ro_ch, name="myro", t=0)
        
    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)
        for _ in range(5):
            self.pulse(ch=cfg['gen_ch'], name="flat1", t=1.0)

In [5]:
config = {'gen_ch': 0, 'ro_ch': 0, 'freq': 100, 'trig_time': 1.0, 'ro_len': 5.0}

prog = MultiPulseProgram(emu.soccfg, reps=1, final_delay=0.5, cfg=config)
artifacts = emu.prepare(prog, soc, memdir=EMU_DIR)

print(f"Artifacts generated in {EMU_DIR}")
print(f"Transaction log: {artifacts['axi_script']}")
map_path = pathlib.Path(EMU_DIR) / "addr_map.json"

# Now you can save it
emu.addrmap.save(map_path)

print(f"Address Map saved to: {map_path}")

Artifacts generated in emulator_output
Transaction log: emulator_output/axi_replay.jsonl
Address Map saved to: emulator_output/addr_map.json
